# Multi-Branch Keras — Modality-Specific Sequence Classifier

This notebook uses a multi-branch deep learning architecture where each sensor modality (Acceleration, Rotation, TOF, Thermopile, STFT, CWT) has its own dedicated branch. 

It follows the exact style of the Random Forest notebook:
- configuration
- data loading
- split
- estimator (Keras Multi-Branch)
- parameter search (grid or Bayesian)
- holdout evaluation
- save results

In [1]:
import os
import sys
import warnings
import json
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Robust Kaggle path routing (handles nested folders from zip uploads)
if os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        # If we find our custom modules in this directory, add it to sys.path
        if 'data_utils.py' in files or 'base_utils_qwen.py' in files or 'multibranch_attention.py' in files:
            if root not in sys.path:
                sys.path.insert(0, root)

from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

import data_utils
import multibranch_attention as multibranch_keras
importlib.reload(multibranch_keras)
from base_utils_qwen import (
    SequenceExtractor,
    SensorAugmentor,
    evaluate_holdout,
    prepare_bayesian_space
)
import matplotlib.pyplot as plt
from pathlib import Path

print('Imports loaded successfully')

Imports loaded successfully


In [2]:
experiment_name = 'multibranch_attention'

# Supported targets: 'bfrb', 'gesture_action', 'orientation', 'gesture', 'gesture_position'
TARGET_COL = 'bfrb'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'

search_mode = 'bayesian'  # 'grid' or 'bayesian'
random_state = 42
n_splits = 1              # 1 -> GroupShuffleSplit; >=2 -> GroupKFold

train_size = 0.8
n_iter = 137
verbose = 4
error_score =  np.nan
n_jobs = 1                 # TensorFlow models should not be fitted in parallel

orientation_filter_list = [] # 'Seated Lean Non Dom - FACE DOWN', 'Lie on Side - Non Dominant', 'Seated Straight', 'Lie on Back'
gesture_position_filter_list = [] # 'Neck', 'Above ear', 'Cheek', 'Eyelash', 'Eyebrow', 'Forehead'
gesture_filter_list = [] # 'Cheek - pinch skin', 'Eyelash - pull hair', 'Eyebrow - pull hair', 'Forehead - scratch', 'Forehead - pull hairline'

problematic_sequence_bool = True

target_only_bool = True

results_dir = Path(workspace_root) / f'results_{experiment_name}_{TARGET_COL}_{search_mode}'
results_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=(1 - train_size),
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)

In [3]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

train_demo_path = data_root / 'train_demographics.csv'
train_demo_df = pd.read_csv(train_demo_path) if train_demo_path.exists() else None

# Derive all supported sequence targets using the same definitions as random_forest.ipynb.
train_df = raw_train_df.copy()
train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')
train_df['gesture_position'] = train_df['gesture'].str.split(' - ').str[0]
train_df['gesture_action'] = train_df['gesture'].str.split(' - ').str[-1]

if TARGET_COL not in {'bfrb', 'gesture_action', 'orientation', 'gesture', 'gesture_position'}:
    raise ValueError("TARGET_COL must be one of 'bfrb', 'gesture_action', 'orientation', or 'gesture'.")

if orientation_filter_list:
    train_df = train_df[train_df['orientation'].isin(orientation_filter_list)].copy()

if gesture_position_filter_list:
    train_df = train_df[train_df['gesture_position'].isin(gesture_position_filter_list)].copy()

if gesture_filter_list:
    train_df = train_df[train_df['gesture'].isin(gesture_filter_list)].copy()

if target_only_bool:
    train_df = train_df[train_df['is_target'] == 1].copy()

train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)
print(f'Target: {TARGET_COL}; sequences after filters: {train_df["sequence_id"].nunique()}')

Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data


Using train.csv: 8151 sequences
Target: bfrb; sequences after filters: 5113


In [4]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=(1 - train_size),
        random_state=random_state,
    )
except Exception as split_error:
    print(f'Balanced split unavailable ({split_error}); using grouped random split.')
    sequence_df = (
        train_df[['sequence_id', 'is_target', TARGET_COL]]
        .drop_duplicates('sequence_id')
        .sort_values('sequence_id')
    )
    splitter = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(sequence_df, groups=sequence_df['sequence_id']))
    train_ids = sequence_df.iloc[train_idx]['sequence_id']
    test_ids = sequence_df.iloc[test_idx]['sequence_id']
    train_sample_df = train_df[train_df['sequence_id'].isin(train_ids)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_ids)].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()
y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()
groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Holdout sequences:', X_test['sequence_id'].nunique())

Train: 3856 seqs | 75.4%
Test:  629 seqs  | 12.3%
Train sequences: 3856
Holdout sequences: 629


In [5]:
extractor = SequenceExtractor(
    output_format='chunks',
    acc_modes='smoothed|velocity|displacement|jerk',

)

clf = multibranch_keras.MultiBranchSequenceClassifier(
    extractor=extractor,
    primary_target=TARGET_COL,
    random_state=random_state,
)

pipeline = Pipeline([
    ('augmentor', SensorAugmentor(
        sequence_col='sequence_id',
        counter_col='sequence_counter'
    )),
    ('estimator', clf),
])

In [6]:
# ============================================================
# Branch filter / depth candidates (unchanged)
# ============================================================

SEARCH_BRANCH_FILTERS = [
    {'acc': 16, 'rotation': 16, 'tof': 16, 'thermo': 16, 'stft': 16, 'cwt': 16},
    {'acc': 32, 'rotation': 32, 'tof': 32, 'thermo': 32, 'stft': 16, 'cwt': 16},
    {'acc': 64, 'rotation': 64, 'tof': 64, 'thermo': 32, 'stft': 64, 'cwt': 64},
    {'acc': 128, 'rotation': 128, 'tof': 128, 'thermo': 32, 'stft': 128, 'cwt': 128},
    {'acc': 200, 'rotation': 200, 'tof': 200, 'thermo': 32, 'stft': 256, 'cwt': 256},
]

SEARCH_BRANCH_DEPTHS = [
    {'acc': 2, 'rotation': 2, 'tof': 2, 'thermo': 1, 'stft': 1, 'cwt': 1},
    {'acc': 4, 'rotation': 4, 'tof': 4, 'thermo': 2, 'stft': 4, 'cwt': 4},
]

SEARCH_CUSTOM_FILTERS = [
    {
        'acc': [32, 64, 64],
        'rotation': [32, 64, 64],
        'tof': [32, 64],
        'thermo': [32],
        'stft': [32],
        'cwt': [32],
    },
]

SEARCH_CONSTANT_FILTERS = [
    {'acc': 16, 'rotation': 16, 'tof': 16, 'thermo': 16, 'stft': 16, 'cwt': 16},
    {'acc': 32, 'rotation': 32, 'tof': 32, 'thermo': 32, 'stft': 16, 'cwt': 16},
    {'acc': 64, 'rotation': 64, 'tof': 64, 'thermo': 32, 'stft': 64, 'cwt': 64},
    {'acc': 128, 'rotation': 128, 'tof': 128, 'thermo': 32, 'stft': 128, 'cwt': 128},
    {'acc': 200, 'rotation': 200, 'tof': 200, 'thermo': 32, 'stft': 256, 'cwt': 256},
]

branch_filter_mode = 'constant'  # 'custom', 'double', or 'constant'

if branch_filter_mode == 'custom':
    branch_filter_candidates = SEARCH_CUSTOM_FILTERS
    branch_depth_candidates = [{
        'acc': 3, 'rotation': 3, 'tof': 2,
        'thermo': 1, 'stft': 1, 'cwt': 1,
    }]
else:
    if branch_filter_mode == 'double':
        branch_filter_candidates = SEARCH_BRANCH_FILTERS
    elif branch_filter_mode == 'constant':
        branch_filter_candidates = SEARCH_CONSTANT_FILTERS
    else:
        raise ValueError("branch_filter_mode must be 'custom', 'double', or 'constant'.")
    branch_depth_candidates = SEARCH_BRANCH_DEPTHS


# ============================================================
# STFT / CWT config-list candidates
# ============================================================

STFT_CONFIGS = [
    None,
    # [{'name': 'stft_default', 'nperseg': 64, 'noverlap': 32}],
    # [
    #     {'name': 'stft_fast', 'nperseg': 32,  'noverlap': 16},
    #     {'name': 'stft_mid',  'nperseg': 64,  'noverlap': 32},
    #     {'name': 'stft_slow', 'nperseg': 128, 'noverlap': 64},
    # ],
]

CWT_CONFIGS = [
    None,
    # [{'name': 'cwt_morl', 'wavelet': 'morl', 'max_scale': 64, 'n_scales': 32}],
    # [
    #     {'name': 'cwt_morl',  'wavelet': 'morl',  'max_scale': 64, 'n_scales': 32},
    #     {'name': 'cwt_mexh',  'wavelet': 'mexh',  'max_scale': 64, 'n_scales': 32},
    #     {'name': 'cwt_gaus2', 'wavelet': 'gaus2', 'max_scale': 64, 'n_scales': 32},
    # ],
]


# ============================================================
# FULL GRID / BAYESIAN PARAM SPACE
# ============================================================

GRID_PARAM_SPACE = {

    # ----------------------------------------------------------
    # AUGMENTOR  (every knob now searchable)
    # ----------------------------------------------------------
    'augmentor__prob':                [0.0, 0.3],
    'augmentor__per_aug_prob':        [0.0, 0.3],
    'augmentor__augmentations': [
        None,
        ['jitter', 'scaling', 'time_shift'],
        ['jitter', 'scaling', 'time_shift', 'channel_dropout'],
        ['jitter', 'scaling', 'time_shift', 'channel_dropout', 'temporal_mask'],
    ],
    'augmentor__sensor_drop_prob':    [0.1, 0.3],
    'augmentor__noise_std':           [0.0, 0.1],
    'augmentor__jitter_sigma':        [0.05, 0.1, 0.2],
    'augmentor__scaling_sigma':       [0.1, 0.2],
    'augmentor__channel_drop_prob':   [0.0, 0.1, 0.3],
    'augmentor__time_shift_frac':     [0.1, 0.2],
    'augmentor__temporal_num_masks':  [1, 2],
    'augmentor__temporal_mask_frac':  [0.05, 0.1],
    'augmentor__warp_sigma':          [0.1, 0.2, 0.3],
    'augmentor__warp_num_knots':      [2, 4, 6],
    'augmentor__crop_frac_range':     [(0.5, 0.9), (0.7, 0.95), (0.4, 0.8)],

    # ----------------------------------------------------------
    # EXTRACTOR — accelerometer
    # ----------------------------------------------------------
    'estimator__extractor__acc_modes': [
        'smoothed|velocity|displacement|jerk',
        'raw|smoothed|velocity|displacement|jerk',
        'smoothed|jerk',
        # None,
    ],
    'estimator__extractor__use_acc_magnitude':         [True, False],
    'estimator__extractor__use_linear_acc_magnitude':  [True, False],
    'estimator__extractor__linear_acc_mode':           [None, 'baseline'],
    'estimator__extractor__use_highpass_fallback':     [True, False],

    # ----------------------------------------------------------
    # EXTRACTOR — rotation
    # ----------------------------------------------------------
    'estimator__extractor__rotation_modes': [
        None,
        # 'quaternion|euler|angular_velocity',
        # 'euler|angular_velocity',
        # 'quaternion',
        # 'rot6d',
    ],
    'estimator__extractor__fix_quaternion_sign': [True],

    # ----------------------------------------------------------
    # EXTRACTOR — ToF
    # ----------------------------------------------------------
    'estimator__extractor__tof_modes':      ['pooled_stats|sensor_stats', 'sensor_stats', 'raw'],
    'estimator__extractor__tof_fill_mode':  ['far_255', 'far_500', 'nan_interpolate', 'zero'],
    'estimator__extractor__tof_n_sensors':  [3, 5],

    # ----------------------------------------------------------
    # EXTRACTOR — thermo
    # ----------------------------------------------------------
    'estimator__extractor__thm_modes': ['centered_diff', 'centered', 'diff', 'raw'],

    # ----------------------------------------------------------
    # EXTRACTOR — signal cleaning / motion / frame stats
    # ----------------------------------------------------------
    'estimator__extractor__frame_stats':            [None, 
                                                    #  'mean,std,min,max,last,first,rms'
                                                     ],
    'estimator__extractor__motion_filter_mode':     [None, 'kalman'],
    'estimator__extractor__use_dead_reckoning':     [False, True],
    'estimator__extractor__dead_reckoning_detrend': [False, True],
    'estimator__extractor__kalman_process_noise':       [1e-4],
    'estimator__extractor__kalman_measurement_noise':   [1e-2],
    'estimator__extractor__window_size':   [5],
    'estimator__extractor__smooth_alpha':  [None, 0.1, 0.3, 0.5],
    'estimator__extractor__clip_value':    [None, 5.0, 10.0, 20.0],
    'estimator__extractor__interp_mode':   ['linear', 'ffill'],

    # ----------------------------------------------------------
    # EXTRACTOR — chunking
    # ----------------------------------------------------------
    'estimator__extractor__output_format':      ['chunks'],
    'estimator__extractor__padding_value':      [0.0],
    'estimator__extractor__maxlen':             [100, 160, 240],
    'estimator__extractor__chunk_window_size':  [50, 100, 120, 150, 200],
    'estimator__extractor__chunk_stride':       [25, 50, 100],
    'estimator__extractor__add_global_context': [True, False],
    'estimator__extractor__resample_modalities':[False, True],
    'estimator__extractor__compute_dt':         [True],

    # ----------------------------------------------------------
    # EXTRACTOR — sampling rates
    # ----------------------------------------------------------
    'estimator__extractor__imu_native_sampling_rate': [100],
    'estimator__extractor__rot_native_sampling_rate': [100],
    'estimator__extractor__tof_native_sampling_rate': [20],
    'estimator__extractor__thm_native_sampling_rate': [20],
    'estimator__extractor__imu_target_sampling_rate': [100],
    'estimator__extractor__rot_target_sampling_rate': [100],
    'estimator__extractor__tof_target_sampling_rate': [20],
    'estimator__extractor__thm_target_sampling_rate': [20],

    # ----------------------------------------------------------
    # EXTRACTOR — STFT (config-list AND scalar paths)
    # ----------------------------------------------------------
    'estimator__extractor__stft_configs':       STFT_CONFIGS,
    # 'estimator__extractor__stft_nperseg':       [32, 64, 128],
    # 'estimator__extractor__stft_noverlap':      [16, 32, 64],
    # 'estimator__extractor__stft_window_type':   ['hann', 'hamming', 'blackman'],
    # 'estimator__extractor__stft_scaling':       ['density', 'spectrum'],
    # 'estimator__extractor__stft_detrend':       ['constant', 'linear', False],
    # 'estimator__extractor__stft_use_log_scale': [True, False],
    # 'estimator__extractor__stft_frequency_bands': [
    #     None,
    #     {'ultra_low': (0.0, 1.0), 'low': (1.0, 3.0), 'mid': (3.0, 6.0), 'high': (6.0, 10.0)},
    # ],

    # ----------------------------------------------------------
    # EXTRACTOR — CWT (config-list AND scalar paths)
    # ----------------------------------------------------------
    'estimator__extractor__cwt_configs':       CWT_CONFIGS,
    # 'estimator__extractor__cwt_wavelet':       ['morl', 'mexh', 'gaus2'],
    # 'estimator__extractor__cwt_max_scale':     [64, 128, 256],
    # 'estimator__extractor__cwt_n_scales':      [16, 32, 64],
    # 'estimator__extractor__cwt_use_log_scale': [True, False],

    # ----------------------------------------------------------
    # EXTRACTOR — problematic-sequence filter
    # ----------------------------------------------------------
    'estimator__extractor__filter_problematic_sequences':   [problematic_sequence_bool],
    'estimator__extractor__ideal_skew_threshold':           [1.0, 1.2, 1.4, 1.6],
    'estimator__extractor__problematic_features_threshold': [2, 3, 4, 5, 6, 7],
    'estimator__extractor__problematic_feature_cols': [
        # None,
        ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_w', 'rot_z'],
    ],

    # ----------------------------------------------------------
    # MODEL — branch architecture
    # ----------------------------------------------------------
    'estimator__branch_filters':          branch_filter_candidates,
    'estimator__branch_num_conv_layers':  branch_depth_candidates,
    'estimator__branch_filter_mode':      [branch_filter_mode],
    'estimator__branch_kernel_sizes': [
        {'acc': 3,  'rotation': 3,  'tof': 3, 'thermo': 3, 'stft': 3, 'cwt': 3},
        {'acc': 5,  'rotation': 5,  'tof': 3, 'thermo': 3, 'stft': 3, 'cwt': 3},
        {'acc': 7,  'rotation': 7,  'tof': 7, 'thermo': 3, 'stft': 3, 'cwt': 3},
        {'acc': 11, 'rotation': 11, 'tof': 11, 'thermo': 3, 'stft': 3, 'cwt': 3},
    ],
    'estimator__branch_pool_sizes': [
        {'acc': 1, 'rotation': 1, 'tof': 1, 'thermo': 1, 'stft': 1, 'cwt': 1},
        {'acc': 2, 'rotation': 2, 'tof': 2, 'thermo': 2, 'stft': 2, 'cwt': 2},
        {'acc': 3, 'rotation': 3, 'tof': 3, 'thermo': 2, 'stft': 2, 'cwt': 2},
    ],
    'estimator__branch_dense_units':  [32, 64, 128],
    'estimator__static_branch_units':  [16, 32, 64],

    # ----------------------------------------------------------
    # MODEL — head, regularization, activation, pooling
    # ----------------------------------------------------------
    'estimator__conv_dropout':    [0.0, 0.1, 0.15, 0.2, 0.3, 0.4],
    'estimator__static_dropout':  [0.0, 0.15, 0.2, 0.3, 0.5],
    'estimator__dense_units':     [32, 64, 128],
    'estimator__dropout':         [0.0, 0.2, 0.3, 0.5],
    'estimator__use_batchnorm':   [True, False],
    'estimator__activation':      ['relu', 'gelu', 'swish', 'elu', 'leaky'],
    'estimator__pooling_mode':    ['attention', 'mean', 'max', 'last'],

    # ----------------------------------------------------------
    # MODEL — optimizer
    # ----------------------------------------------------------
    'estimator__optimizer_name':      ['adam', 'adamw', 'rmsprop', 'sgd', 'nadam', 'adagrad', 'adadelta'],
    'estimator__learning_rate':       [1e-4, 5e-4, 1e-3, 3e-3],
    'estimator__weight_decay':        [0.0, 1e-5, 1e-4, 1e-3],
    'estimator__momentum':            [0.5, 0.9, 0.99],
    'estimator__gradient_clip_norm':  [None, 0.5, 1.0, 5.0],
    'estimator__gradient_clip_value': [None, 0.5, 1.0],

    # ----------------------------------------------------------
    # MODEL — loss
    # ----------------------------------------------------------
    'estimator__loss_name':       ['sparse_cce', 'focal'],
    'estimator__label_smoothing': [0.0, 0.05, 0.1],

    # ----------------------------------------------------------
    # TRAINING LOOP
    # ----------------------------------------------------------
    'estimator__epochs':                  [100, 150, 200],
    'estimator__batch_size':              [32, 64, 128, 200, 256],
    'estimator__validation_split':        [0.1, 0.15, 0.2, 0.25],
    'estimator__early_stopping_patience': [5, 10, 15, 20],
    'estimator__class_weight_mode':       ['balanced'],

    # ----------------------------------------------------------
    # LR SCHEDULER
    # ----------------------------------------------------------
    'estimator__use_lr_scheduler':   [True],
    'estimator__lr_scheduler_type':  ['plateau', 'cosine'],
    'estimator__lr_factor':          [0.3, 0.5, 0.7],
    'estimator__lr_patience':        [3, 5, 10],
    'estimator__min_lr':             [1e-7, 1e-6, 1e-5],
    'estimator__cosine_t_max':       [50, 100, 200],
    'estimator__warmup_epochs':      [0, 3, 5],
}


# ============================================================
# Build and validate
# ============================================================

if SKOPT_AVAILABLE:
    BAYESIAN_PARAM_SPACE = multibranch_keras.prepare_multibranch_bayesian_space(
        GRID_PARAM_SPACE
    )
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

for space_name, candidate_space in (
    ('grid', GRID_PARAM_SPACE),
    ('bayesian', BAYESIAN_PARAM_SPACE),
):
    unknown_params = set(candidate_space) - set(pipeline.get_params(deep=True))
    if unknown_params:
        raise ValueError(
            f'Unknown {space_name} search parameters: {sorted(unknown_params)}'
        )

param_space = BAYESIAN_PARAM_SPACE if search_mode == 'bayesian' else GRID_PARAM_SPACE

print(f'Validated {len(GRID_PARAM_SPACE)} Grid and Bayesian search parameters.')
print(f'Total reachable params in pipeline: {len(pipeline.get_params(deep=True))}')

Validated 91 Grid and Bayesian search parameters.
Total reachable params in pipeline: 117


In [7]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=None,  # Pipeline delegates to sequence-level estimator.score().
        cv=cv_object,
        n_jobs=n_jobs,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=None,
        cv=cv_object,
        n_jobs=n_jobs,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )

search.fit(X_train, y_train, groups=groups)

print('Best CV score:', search.best_score_)
print('Best params:', search.best_params_)

Fitting 1 folds for each of 1 candidates, totalling 1 fits
[CV 1/1] END augmentor__augmentations=["jitter", "scaling", "time_shift"], augmentor__channel_drop_prob=0.3, augmentor__crop_frac_range=[0.4, 0.8], augmentor__jitter_sigma=0.05, augmentor__noise_std=0.1, augmentor__per_aug_prob=0.0, augmentor__prob=0.0, augmentor__scaling_sigma=0.2, augmentor__sensor_drop_prob=0.1, augmentor__temporal_mask_frac=0.1, augmentor__temporal_num_masks=2, augmentor__time_shift_frac=0.1, augmentor__warp_num_knots=2, augmentor__warp_sigma=0.2, estimator__activation=relu, estimator__batch_size=200, estimator__branch_dense_units=128, estimator__branch_filter_mode=constant, estimator__branch_filters={"acc": 16, "cwt": 16, "rotation": 16, "stft": 16, "thermo": 16, "tof": 16}, estimator__branch_kernel_sizes={"acc": 11, "cwt": 3, "rotation": 11, "stft": 3, "thermo": 3, "tof": 11}, estimator__branch_num_conv_layers={"acc": 2, "cwt": 1, "rotation": 2, "stft": 1, "thermo": 1, "tof": 2}, estimator__branch_pool_si

ValueError: 
All the 1 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\maran\anaconda3\envs\ds_kaggle\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\maran\anaconda3\envs\ds_kaggle\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\maran\anaconda3\envs\ds_kaggle\lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
  File "c:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\src\multibranch_attention.py", line 591, in fit
    self.model_ = build_mu

In [ ]:

# --- pull the history out of the fitted pipeline ---
best_estimator = best_model.named_steps['estimator']
history = best_estimator.get_history_dict()   # returns self.history_

loss      = history.get('loss', [])
val_loss  = history.get('val_loss', [])
acc       = history.get('accuracy', [])
val_acc   = history.get('val_accuracy', [])

has_val = len(val_loss) > 0 and len(val_acc) > 0
epochs_ran = range(1, len(loss) + 1)

# --- two subplots, two lines each ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ---- Loss ----
ax = axes[0]
ax.plot(epochs_ran, loss, label='train loss', linewidth=2)
if has_val:
    ax.plot(epochs_ran, val_loss, label='val loss', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training vs Validation Loss')
ax.grid(alpha=0.3)
ax.legend()

# ---- Accuracy ----
ax = axes[1]
ax.plot(epochs_ran, acc, label='train acc', linewidth=2)
if has_val:
    ax.plot(epochs_ran, val_acc, label='val acc', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Training vs Validation Accuracy')
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()

# save next to the other results (save_dir is defined in your last cell)
curves_path = save_dir / f'{experiment_name}_curves.png'
# plt.savefig(curves_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Saved curves to {curves_path}')
print(f'Epochs run: {len(loss)}')
if has_val:
    best_epoch = int(np.argmin(val_loss)) + 1
    print(f'Best val_loss epoch: {best_epoch}  (val_loss={min(val_loss):.4f})')

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score / macro F1:', eval_results['competition_score'])

In [ ]:
save_dir = results_dir / timestamp
save_dir.mkdir(parents=True, exist_ok=True)
best_estimator = best_model.named_steps['estimator']
best_estimator.model_.save(save_dir / 'best_model.h5')

pd.DataFrame(search.cv_results_).to_csv(
    save_dir / f'{experiment_name}_cv.csv',
    index=False,
)
eval_results['results_df'].to_csv(
    save_dir / f'{experiment_name}_holdout.csv',
    index=False,
)
pd.DataFrame([{
    'target': TARGET_COL,
    'best_score': search.best_score_,
    'best_params': json.dumps(search.best_params_, default=str),
    'holdout_score': eval_results['competition_score'],
}]).to_csv(
    save_dir / f'{experiment_name}_best.csv',
    index=False,
)

with open(save_dir / 'results.json', 'w') as results_file:
    json.dump({
        'target': TARGET_COL,
        'best_score': float(search.best_score_),
        'best_params': search.best_params_,
        'holdout_score': float(eval_results['competition_score']),
        'random_state': random_state,
        'train_sequences': int(X_train['sequence_id'].nunique()),
        'holdout_sequences': int(X_test['sequence_id'].nunique()),
    }, results_file, indent=2, default=str)

print(f'Results saved to {save_dir}')